# 用 Sciverse 做科学 RAG 数据源

> 将 Sciverse 作为 RAG pipeline 的检索后端，为 LLM 提供可信科学证据

**场景**: 开发者构建科学问答系统或 RAG 应用，需要从权威学术文献中检索证据来 ground LLM 的回答，避免幻觉。

**使用接口**: agentic-search, content

**预估调用量**: ~5–15 次 API 调用 / 一次 RAG 查询

---


## Step 1: 环境准备

安装依赖并配置环境变量


In [ ]:
!pip install httpx openai
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # 替换为你的真实值


## Step 2: 调用 agentic-search 获取证据

一次调用即可获得经过打分的文献片段


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def sciverse_retrieve(query: str, top_k: int = 10):
    """调用 agentic-search 获取相关文献片段"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/agentic-search",
            headers=HEADERS,
            json={"query": query, "top_k": top_k}
        )
        resp.raise_for_status()
        data = resp.json()
        return [
            {"text": h["chunk"], "doc_id": h["doc_id"],
             "title": h["title"], "score": h["score"]}
            for h in data["hits"]
        ]


## Step 3: 证据过滤

按 score 阈值过滤低质量片段


In [ ]:
def filter_evidence(hits: list, threshold: float = 0.65) -> list:
    """过滤低分片段，按 score 降序排列"""
    filtered = [h for h in hits if h["score"] >= threshold]
    return sorted(filtered, key=lambda x: x["score"], reverse=True)

async def main():
    hits = await sciverse_retrieve("mRNA LNP delivery system improvements")
    top_evidence = filter_evidence(hits, threshold=0.65)
    print(f"Retrieved {len(hits)} chunks, filtered to {len(top_evidence)} high-quality")
    return top_evidence

top_evidence = asyncio.run(main())


## Step 4: 基于证据生成 Grounded Answer（可选增强）

将证据注入 LLM prompt，生成带引用的回答。此步骤依赖 OpenAI API Key，非 Sciverse 必需


In [ ]:
from openai import OpenAI

client = OpenAI()  # 自动读取 OPENAI_API_KEY

context = "\
\
".join([
    f"[{i+1}] {e['title']}\
{e['text']}"
    for i, e in enumerate(top_evidence[:5])
])

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "基于提供的文献证据回答问题。每个论点用 [编号] 标注来源。如果证据不足，说明无法确定。不要编造未在证据中出现的信息。"},
        {"role": "user", "content": f"问题：mRNA LNP 递送系统最新改进？\
\
证据：\
{context}"}
    ]
)
print(resp.choices[0].message.content)


## 注意事项

- agentic-search 的 top_k 范围为 1–100，RAG 场景建议 10–20
- score 阈值建议 0.6–0.7，过低会引入噪声，过高可能丢失相关证据
- 生产环境建议缓存高频查询结果，减少 API 调用和延迟
- 如需更精确的证据，可对 top hits 再调用 content(doc_id, offset) 接口获取完整段落上下文


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
